In [12]:
"""
Dominant Residential Building Type at District Level in Bavaria
+ Percentage shares per building type per district
+ Bar chart of class distribution (absolute number of buildings)

IMPORTANT:
- The expensive building assignment is cached.
- For plot adjustments, load cache only without recomputation.
"""

import math
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FuncFormatter
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    from matplotlib_map_utils import north_arrow, scale_bar
    HAS_MMU = True
except Exception:
    HAS_MMU = False

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 13,
    'axes.titlesize': 19,
    'axes.labelsize': 13,
    'legend.fontsize': 13,
    'legend.title_fontsize': 14,
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
})

# ============================================================================
# CONFIGURATION - Column Names and Paths
# ============================================================================

# Column names
COL_GEOMETRY = 'geometry'
COL_RES_SUBCLASS = 'res_subclass'
COL_ARS = 'Regionalschlüssel_ARS'
COL_GEN = 'GeografischerName_GEN'
COL_EWZ = 'Einwohnerzahl_EWZ'
COL_LK_KEY = 'LK_KEY'
COL_LK_NAME = 'LK_NAME'

# Building types
BUILDING_TYPES = ['SFH-DB', 'SBD', 'TB', 'MFH-AB', 'unclassified res']

# Paths
PATH_BUILDINGS = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_residential_types_thr_h11m.gpkg"
PATH_LANDKREISE = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
LAYER_LANDKREISE = 'v_vg250_krs'
PATH_OUTPUT = r"C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen"

# Cache paths (to allow plots without recomputation)
PATH_CACHE_DIR = Path(PATH_OUTPUT) / 'cache_landkreis_buildingtypes'
PATH_CACHE_STATS = PATH_CACHE_DIR / 'landkreis_buildingtype_stats.csv'
PATH_CACHE_GPKG = PATH_CACHE_DIR / 'landkreis_buildingtype_map.gpkg'
CACHE_LAYER_NAME = 'landkreis_stats'

# Control
REBUILD_CACHE = False
MAKE_MAP = True
MAKE_BAR = True

# Bavaria filter: ARS numbers starting with 09
BAYERN_ARS_PREFIX = '09'

# Color scheme
COLOR_MAP_TYPES = {
    'SFH-DB': "#82cbec",            # Blue
    'SBD': "#febd2b",               # Yellow
    'TB': "#9aab4b",                # Green
    'MFH-AB': "#d94f21",            # Red
    'unclassified res': "#b3b3b3",  # Gray
}

# Template settings
TARGET_CRS = 'EPSG:25832'
CRS_NOTE = 'CRS: EPSG:25832 - ETRS89 / UTM zone 32N'
GRID_STEP_M = 50000
MAP_PADDING_M = 10000
LABEL_OFFSET_M = 10000
LABEL_CLAMP_MARGIN_M = 7000

# ============================================================================
# FUNCTIONS
# ============================================================================

def read_admin_data():
    """Read district data; try desired layer, otherwise fallback."""
    try:
        gdf_admin = gpd.read_file(PATH_LANDKREISE, layer=LAYER_LANDKREISE)
        print(f"  Layer loaded: {LAYER_LANDKREISE}")
        return gdf_admin
    except Exception as e:
        print(f"  Notice: Layer '{LAYER_LANDKREISE}' not readable ({e})")
        print("  Fallback: reading default layer from file")
        return gpd.read_file(PATH_LANDKREISE)


def load_data():
    """Load district layer for Bavaria and filtered building data."""
    print("Loading administrative data (district layer)...")
    gdf_admin = read_admin_data()
    print(f"  Total units: {len(gdf_admin)}")

    # Find ARS column
    if COL_ARS in gdf_admin.columns:
        ars_col = COL_ARS
    else:
        available_cols = [c for c in gdf_admin.columns if 'ARS' in c or 'ars' in c]
        if not available_cols:
            raise ValueError('No ARS column found.')
        ars_col = available_cols[0]
        print(f"  Using ARS column: {ars_col}")

    # Filter Bavaria
    gdf_bayern = gdf_admin[gdf_admin[ars_col].astype(str).str.startswith(BAYERN_ARS_PREFIX)].copy()
    gdf_bayern = gdf_bayern.reset_index(drop=True)
    print(f"  Bavaria districts: {len(gdf_bayern)}")

    # District key: first 5 characters of ARS
    gdf_bayern[COL_LK_KEY] = gdf_bayern[ars_col].astype(str).str[:5]

    # Name column for map labels
    gdf_bayern[COL_LK_NAME] = gdf_bayern[COL_GEN].astype(str) if COL_GEN in gdf_bayern.columns else gdf_bayern[COL_LK_KEY]

    # Columns to carry through dissolve
    keep_cols = [COL_LK_KEY, COL_LK_NAME, COL_GEOMETRY]
    aggfunc = {COL_LK_NAME: 'first'}
    if COL_GEN in gdf_bayern.columns:
        keep_cols.append(COL_GEN)
        aggfunc[COL_GEN] = 'first'
    if COL_EWZ in gdf_bayern.columns:
        keep_cols.append(COL_EWZ)
        aggfunc[COL_EWZ] = 'sum'

    gdf_lk = gdf_bayern[keep_cols].dissolve(by=COL_LK_KEY, aggfunc=aggfunc, as_index=False)
    print(f"  Unique districts in Bavaria: {len(gdf_lk)}")
    print(f"  Columns: {list(gdf_lk.columns)}")

    bbox = tuple(gdf_lk.total_bounds)
    print(f"  Bavaria Bounding Box: {bbox}")

    print("\nLoading building data (filtered to Bavaria bbox)...")
    gdf_buildings = gpd.read_file(
        PATH_BUILDINGS,
        bbox=bbox,
        columns=[COL_RES_SUBCLASS]
    )
    print(f"  Buildings loaded: {len(gdf_buildings)}")

    if gdf_buildings.crs != gdf_lk.crs:
        print(f"  Converting building CRS: {gdf_buildings.crs} -> {gdf_lk.crs}")
        gdf_buildings = gdf_buildings.to_crs(gdf_lk.crs)

    return gdf_buildings, gdf_lk


def spatial_join_buildings_to_landkreise(gdf_buildings, gdf_lk):
    """Spatial join: buildings to districts."""
    print("\nSpatial join at district level...")
    gdf_joined = gpd.sjoin(
        gdf_buildings[[COL_GEOMETRY, COL_RES_SUBCLASS]],
        gdf_lk[[COL_LK_KEY, COL_LK_NAME, COL_GEOMETRY]],
        how='inner',
        predicate='intersects'
    )
    gdf_joined = gdf_joined[~gdf_joined.index.duplicated(keep='first')]
    print(f"  Buildings joined to district: {len(gdf_joined)}")
    return gdf_joined


def compute_stats_per_landkreis(gdf_joined):
    """Compute absolute + percentage values per building type per district + dominant type."""
    print("\nComputing metrics per district...")

    stats = gdf_joined.groupby([COL_LK_KEY, COL_RES_SUBCLASS]).size().reset_index(name='count')
    stats_pivot = stats.pivot_table(
        index=COL_LK_KEY,
        columns=COL_RES_SUBCLASS,
        values='count',
        fill_value=0
    ).reset_index()
    stats_pivot.columns.name = None

    # Ensure all types are present
    for t in BUILDING_TYPES:
        if t not in stats_pivot.columns:
            stats_pivot[t] = 0

    type_cols = BUILDING_TYPES.copy()

    # Absolute total and percentage values
    stats_pivot['total_buildings'] = stats_pivot[type_cols].sum(axis=1)
    for t in type_cols:
        stats_pivot[f'{t}_pct'] = (stats_pivot[t] / stats_pivot['total_buildings'] * 100).round(2)

    # Dominant building type
    stats_pivot['dominant_type'] = stats_pivot[type_cols].idxmax(axis=1)
    stats_pivot['dominant_type_count'] = stats_pivot[type_cols].max(axis=1)
    stats_pivot['dominant_type_pct'] = (stats_pivot['dominant_type_count'] / stats_pivot['total_buildings'] * 100).round(2)

    print(f"  Metrics computed for {len(stats_pivot)} districts")
    return stats_pivot


def create_output_dir():
    output_dir = Path(PATH_OUTPUT)
    output_dir.mkdir(parents=True, exist_ok=True)
    PATH_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    return output_dir


def cache_exists():
    return PATH_CACHE_STATS.exists() and PATH_CACHE_GPKG.exists()


def save_cache(stats_lk, gdf_result):
    stats_lk.to_csv(PATH_CACHE_STATS, index=False, encoding='utf-8-sig')
    gdf_result.to_file(PATH_CACHE_GPKG, layer=CACHE_LAYER_NAME, driver='GPKG')
    print(f"\nCache saved:\n  {PATH_CACHE_STATS}\n  {PATH_CACHE_GPKG}")


def load_cache():
    print("\nLoading data from cache...")
    stats_lk = pd.read_csv(PATH_CACHE_STATS, dtype={COL_LK_KEY: str})
    gdf_result = gpd.read_file(PATH_CACHE_GPKG, layer=CACHE_LAYER_NAME)
    print(f"  Cache loaded: {len(stats_lk)} districts")
    return stats_lk, gdf_result


def build_and_cache_from_raw():
    """Execute heavy pipeline once and cache results."""
    gdf_buildings, gdf_lk = load_data()
    gdf_joined = spatial_join_buildings_to_landkreise(gdf_buildings, gdf_lk)
    del gdf_buildings

    stats_lk = compute_stats_per_landkreis(gdf_joined)
    del gdf_joined

    gdf_result = gdf_lk.merge(stats_lk, on=COL_LK_KEY, how='inner')
    save_cache(stats_lk, gdf_result)
    return stats_lk, gdf_result


def add_matplotlib_grid(ax, bounds, step=GRID_STEP_M):
    """Add UTM coordinate grid with crosses."""
    minx, miny, maxx, maxy = bounds
    x_start = math.ceil(minx / step) * step
    x_end = math.floor(maxx / step) * step
    y_start = math.ceil(miny / step) * step
    y_end = math.floor(maxy / step) * step

    x_major = [x_start + i * step for i in range(int((x_end - x_start) / step) + 1)] if x_start <= x_end else []
    y_major = [y_start + i * step for i in range(int((y_end - y_start) / step) + 1)] if y_start <= y_end else []

    ax.set_xticks(x_major)
    ax.set_yticks(y_major)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{int(round(x / 1000))}'))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f'{int(round(y / 1000))}'))

    if x_major and y_major:
        x_pts, y_pts = [], []
        for xv in x_major:
            for yv in y_major:
                x_pts.append(xv)
                y_pts.append(yv)
        ax.scatter(x_pts, y_pts, marker='+', s=30, linewidths=1.0, color='#8a8a8a', alpha=0.95, zorder=4, clip_on=True)

    ax.tick_params(axis='both', which='major', labelsize=11, length=0, colors='#9B9999')


def add_scientific_frame(ax, gdf_base):
    """Add scientific frame with coordinate labels, grid, and background."""
    ax.set_facecolor('#f1f1f1')
    minx, miny, maxx, maxy = gdf_base.total_bounds
    bounds = (minx - MAP_PADDING_M, miny - MAP_PADDING_M, maxx + MAP_PADDING_M, maxy + MAP_PADDING_M)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.margins(0)
    add_matplotlib_grid(ax, bounds)

    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_autoscale_on(False)

    ax.set_xlabel('Easting (km) - UTM 32N', fontsize=12, color='#555555')
    ax.set_ylabel('Northing (km) - UTM 32N', fontsize=12, color='#555555')
    ax.text(0.01, 0.01, CRS_NOTE, transform=ax.transAxes, ha='left', va='bottom', fontsize=11, color='#555555')

    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
    for side in ['left', 'bottom']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color('#636262')
        ax.spines[side].set_linewidth(0.8)


def add_north_arrow(ax):
    """Add north arrow in upper right corner."""
    if HAS_MMU:
        north_arrow(
            ax=ax,
            location='upper right',
            size='md',
            rotation={'degrees': 0},
            aob={
                'bbox_to_anchor': (0.985, 0.985),
                'bbox_transform': ax.transAxes,
                'pad': 0.06,
                'borderpad': 0.06,
                'facecolor': 'none',
                'edgecolor': 'none',
                'alpha': 1.0,
                'frameon': False,
            },
        )
        return

    ax.annotate(
        'N',
        xy=(0.965, 0.97),
        xytext=(0.965, 0.86),
        xycoords='axes fraction',
        textcoords='axes fraction',
        ha='center',
        va='center',
        fontsize=16,
        fontweight='bold',
        arrowprops=dict(arrowstyle='-|>', color='#222222', linewidth=1.4, shrinkA=0, shrinkB=0)
    )


def add_scale_bar(ax):
    """Add scale bar in lower right corner."""
    if HAS_MMU:
        scale_bar(
            ax=ax,
            location='lower right',
            size='xs',
            style='ticks',
            bar={
                'projection': TARGET_CRS,
                'unit': 'km',
                'max': 50,
                'major_div': 2,
                'minor_div': 1,
                'minor_type': 'none',
                'reverse': False,
            },
            labels={
                'labels': ['0', '25', '50'],
                'style': 'major',
                'loc': 'below',
                'fontsize': 9,
            },
            units={'loc': 'text', 'label': 'km'},
            text={'fontfamily': 'sans-serif', 'fontsize': 9, 'textcolor': '#222222'},
            aob={
                'pad': 0.0,
                'borderpad': 1.0,
                'facecolor': 'none',
                'edgecolor': 'none',
                'alpha': 1.0,
                'frameon': False,
            },
        )
        return

    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    span_x = x1 - x0
    span_y = y1 - y0

    total_km = 50
    segment_km = 25
    bar_len = total_km * 1000
    segment_len = segment_km * 1000

    x_start = x1 - span_x * 0.25
    y_start = y0 + span_y * 0.032

    ax.plot([x_start, x_start + bar_len], [y_start, y_start], color='#222222', linewidth=1.3, zorder=5)
    tick_h = span_y * 0.006
    for x in [x_start, x_start + segment_len, x_start + bar_len]:
        ax.plot([x, x], [y_start - tick_h, y_start + tick_h], color='#222222', linewidth=1.0, zorder=5)

    txt_y = y_start + span_y * 0.011
    ax.text(x_start, txt_y, '0', ha='center', va='bottom', fontsize=9, color='#222222')
    ax.text(x_start + segment_len, txt_y, '25', ha='center', va='bottom', fontsize=9, color='#222222')
    ax.text(x_start + bar_len, txt_y, '50 km', ha='center', va='bottom', fontsize=9, color='#222222')


def add_labels_non_sfh(ax, gdf, offset_m=LABEL_OFFSET_M):
    """Label all districts whose dominant type is not SFH-DB with offset positioning."""
    required_cols = {COL_LK_NAME, 'dominant_type', COL_GEOMETRY}
    if not required_cols.issubset(set(gdf.columns)):
        print("  Notice: Required columns for labels missing")
        return

    gdf_labels = gdf[gdf['dominant_type'] != 'SFH-DB'].copy()
    if gdf_labels.empty:
        print("  Notice: No non-SFH-DB districts available for labeling")
        return

    bounds = gdf.total_bounds
    center_x = (bounds[0] + bounds[2]) / 2
    center_y = (bounds[1] + bounds[3]) / 2

    # Keep label anchors inside plotted map frame (with small inner margin)
    x_min = bounds[0] - MAP_PADDING_M + LABEL_CLAMP_MARGIN_M
    x_max = bounds[2] + MAP_PADDING_M - LABEL_CLAMP_MARGIN_M
    y_min = bounds[1] - MAP_PADDING_M + LABEL_CLAMP_MARGIN_M
    y_max = bounds[3] + MAP_PADDING_M - LABEL_CLAMP_MARGIN_M

    # Build normalized name groups to disambiguate equal names (e.g. city vs district)
    def _norm_name(name):
        n = str(name).strip()
        n = n.replace('Landkreis ', '').replace('Lkr. ', '').replace('Stadtkreis ', '')
        n = n.replace('Kreisfreie Stadt ', '').replace('Landeshauptstadt ', '').replace('Stadt ', '')
        n = n.split(',')[0].strip().lower()
        return n

    type_by_idx = {}
    grouped = {}
    for idx, row in gdf_labels.iterrows():
        key = _norm_name(row[COL_LK_NAME])
        grouped.setdefault(key, []).append((idx, row.geometry.area))

    for key, items in grouped.items():
        if len(items) <= 1:
            continue
        # Largest area is typically district, smallest is typically city
        items_sorted = sorted(items, key=lambda x: x[1])
        for i, (idx, _) in enumerate(items_sorted):
            type_by_idx[idx] = 'stadt' if i == 0 else 'landkreis'

    def _format_label(row):
        raw = str(row[COL_LK_NAME]).strip()
        raw_low = raw.lower()
        base = raw.split(',')[0].strip()

        if row.name in type_by_idx:
            return f"Stadt {base}" if type_by_idx[row.name] == 'stadt' else f"Lkr. {base}"

        if 'landkreis' in raw_low or raw_low.startswith('lkr.'):
            return f"Lkr. {base}"
        if 'stadt' in raw_low or 'landeshauptstadt' in raw_low:
            return f"Stadt {base}"
        return base

    # Collision thresholds in data units
    span_x = (bounds[2] + MAP_PADDING_M) - (bounds[0] - MAP_PADDING_M)
    span_y = (bounds[3] + MAP_PADDING_M) - (bounds[1] - MAP_PADDING_M)
    min_dx = span_x * 0.11
    min_dy = span_y * 0.03
    placed = []

    points = gdf_labels.representative_point()
    for (_, row), p in zip(gdf_labels.iterrows(), points):
        dx = p.x - center_x
        dy = p.y - center_y
        norm = math.hypot(dx, dy)
        if norm == 0:
            dx, dy, norm = 1.0, 1.0, math.sqrt(2)
        ux, uy = dx / norm, dy / norm

        # Try multiple directions; if one direction crosses map bounds, switch direction
        direction_candidates = [
            (ux, uy),
            (-ux, -uy),
            (1.0 if ux >= 0 else -1.0, 0.0),
            (-1.0 if ux >= 0 else 1.0, 0.0),
            (0.0, 1.0 if uy >= 0 else -1.0),
            (0.0, -1.0 if uy >= 0 else 1.0),
        ]

        chosen = None
        for cx, cy in direction_candidates:
            cand_tx = p.x + cx * offset_m
            cand_ty = p.y + cy * offset_m
            if x_min <= cand_tx <= x_max and y_min <= cand_ty <= y_max:
                chosen = (cand_tx, cand_ty, cx)
                break

        if chosen is None:
            cand_tx = p.x - ux * offset_m
            cand_ty = p.y - uy * offset_m
            cand_tx = min(max(cand_tx, x_min), x_max)
            cand_ty = min(max(cand_ty, y_min), y_max)
            chosen = (cand_tx, cand_ty, -ux)

        tx, ty, dir_x = chosen

        # Iteratively move labels until there is no overlap with previously placed labels
        label_side = 1 if dir_x >= 0 else -1
        for _ in range(40):
            overlap = any((abs(tx - px) < min_dx and abs(ty - py) < min_dy) for px, py in placed)
            if not overlap:
                break
            ty += (min_dy * 0.55) if ty <= center_y else (-min_dy * 0.55)
            ty = min(max(ty, y_min), y_max)
            tx += label_side * (min_dx * 0.20)
            tx = min(max(tx, x_min), x_max)

        # If still overlapping after retries, skip this label to avoid unreadable collisions
        if any((abs(tx - px) < min_dx and abs(ty - py) < min_dy) for px, py in placed):
            continue

        ha = 'left' if tx >= p.x else 'right'
        label_text = _format_label(row)

        # Avoid left/right clipping by adapting text anchor and x-position
        approx_char_w = span_x * 0.0065
        text_w = max(1, len(label_text)) * approx_char_w
        pad_x = span_x * 0.006
        if tx - text_w < x_min + pad_x:
            ha = 'left'
            tx = min(max(tx, x_min + pad_x), x_max - pad_x)
        elif tx + text_w > x_max - pad_x:
            ha = 'right'
            tx = max(min(tx, x_max - pad_x), x_min + pad_x)

        placed.append((tx, ty))

        ax.annotate(
            label_text,
            xy=(p.x, p.y),
            xytext=(tx, ty),
            textcoords='data',
            ha=ha,
            va='center',
            fontsize=10,
            color='#1f1f1f',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.7),
            arrowprops=dict(arrowstyle='-', color='#8a8a8a', lw=0.7),
            zorder=6,
            clip_on=True,
        )


def add_legend_bottom_right(ax, handles, title):
    """Add legend outside on the right side."""
    ax.legend(
        handles=handles,
        title=title,
        loc='lower left',
        bbox_to_anchor=(1.01, 0.02),
        borderaxespad=0.0,
        frameon=False,
        fontsize=13,
        title_fontsize=14,
        handlelength=2.0,
        labelspacing=0.6,
        borderpad=0.0,
    )


def plot_dominant_type_map(gdf, output_dir):
    """Map: Dominant building type per district with template theme."""
    print("\nCreating map: Dominant building type (district level)...")

    fig, ax = plt.subplots(figsize=(8.6, 9.4))
    fig.subplots_adjust(right=0.80, left=0.08, top=0.92, bottom=0.10)

    colors = gdf['dominant_type'].map(COLOR_MAP_TYPES).fillna('#bdbdbd')
    gdf.plot(ax=ax, color=colors, edgecolor='white', linewidth=0.6, zorder=2)

    present_types = [t for t in BUILDING_TYPES if t in gdf['dominant_type'].unique()]
    legend_elements = [
        mpatches.Patch(facecolor=COLOR_MAP_TYPES[t], edgecolor='white', label=t)
        for t in present_types
    ]

    add_labels_non_sfh(ax, gdf)
    ax.set_title('Dominant Residential Building Type at District Level in Bavaria', fontweight='bold', fontsize=18, pad=14)
    add_scientific_frame(ax, gdf)
    add_north_arrow(ax)
    add_scale_bar(ax)
    add_legend_bottom_right(ax, legend_elements, 'Dominant\nBuilding Type')

    out_file = output_dir / 'dominant_type_bayern_landkreise.jpg'
    plt.savefig(out_file, bbox_inches='tight', facecolor='white', format='jpg')
    print(f"  Saved: {out_file}")

    plt.close()

# ============================================================================
# MAIN PROGRAM
# ============================================================================

print("=" * 72)
print("RESIDENTIAL BUILDING TYPES BAVARIA - DISTRICT LEVEL")
print("=" * 72)

output_dir = create_output_dir()

# 1) Load data from cache or recompute
if (not REBUILD_CACHE) and cache_exists():
    stats_lk, gdf_result = load_cache()
else:
    print("\nCache missing or REBUILD_CACHE=True -> recomputing...")
    stats_lk, gdf_result = build_and_cache_from_raw()

# 2) Generate plots (fast, without heavy computation)
if MAKE_MAP:
    plot_dominant_type_map(gdf_result, output_dir)

# 3) CSV export with absolute + percentage values
csv_file = output_dir / 'landkreis_buildingtype_stats.csv'

stats_export_cols = ['total_buildings', 'dominant_type', 'dominant_type_count', 'dominant_type_pct']
for t in BUILDING_TYPES:
    stats_export_cols.append(t)
for t in BUILDING_TYPES:
    stats_export_cols.append(f'{t}_pct')

export_df = stats_lk[[COL_LK_KEY] + stats_export_cols].copy()

# Merge name and population from gdf_result
for col in [COL_GEN, COL_EWZ]:
    if col in gdf_result.columns:
        export_df = export_df.merge(
            gdf_result[[COL_LK_KEY, col]].drop_duplicates(subset=COL_LK_KEY),
            on=COL_LK_KEY,
            how='left'
        )

# Rename LK_KEY -> ARS, preserve leading zeros
export_df = export_df.rename(columns={COL_LK_KEY: 'ARS'})
export_df['ARS'] = export_df['ARS'].astype(str).str.zfill(5)

# Column order: ARS, name, population, then stats
front_cols = ['ARS']
for col in [COL_GEN, COL_EWZ]:
    if col in export_df.columns:
        front_cols.append(col)
export_df = export_df[front_cols + stats_export_cols]

export_df.to_csv(csv_file, index=False, encoding='utf-8-sig')
print(f"\nCSV saved: {csv_file}")

print("\nDone. Output saved to:")
print(output_dir)
print("\nNote: For plot adjustments, set REBUILD_CACHE=False; then only cache is loaded.")

RESIDENTIAL BUILDING TYPES BAVARIA - DISTRICT LEVEL

Loading data from cache...
  Cache loaded: 96 districts

Creating map: Dominant building type (district level)...
  Saved: C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen\dominant_type_bayern_landkreise.jpg

CSV saved: C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen\landkreis_buildingtype_stats.csv

Done. Output saved to:
C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen

Note: For plot adjustments, set REBUILD_CACHE=False; then only cache is loaded.


In [13]:
"""
Dominant Residential Building Type of NEW BUILDINGS SINCE 2015 at District Level in Bavaria
+ Percentage shares per building type per district
+ Bar chart of class distribution (absolute number of buildings)

IMPORTANT:
- The expensive building assignment is cached.
- For plot adjustments, load cache only without recomputation.
"""

import math
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FuncFormatter
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    from matplotlib_map_utils import north_arrow, scale_bar
    HAS_MMU = True
except Exception:
    HAS_MMU = False

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 13,
    'axes.titlesize': 19,
    'axes.labelsize': 13,
    'legend.fontsize': 13,
    'legend.title_fontsize': 14,
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
})

# ============================================================================
# CONFIGURATION - Column Names and Paths
# ============================================================================

# Column names
COL_GEOMETRY = 'geometry'
COL_RES_SUBCLASS = 'res_subclass'
COL_ARS = 'Regionalschlüssel_ARS'
COL_GEN = 'GeografischerName_GEN'
COL_EWZ = 'Einwohnerzahl_EWZ'
COL_LK_KEY = 'LK_KEY'
COL_LK_NAME = 'LK_NAME'

# Building types
BUILDING_TYPES = ['SFH-DB', 'SBD', 'TB', 'MFH-AB', 'unclassified res']

# Paths
PATH_BUILDINGS = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_new_buildings_thr_h11m.gpkg"
PATH_LANDKREISE = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
LAYER_LANDKREISE = 'v_vg250_krs'
PATH_OUTPUT = r"C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen"

# Cache paths (to allow plots without recomputation)
PATH_CACHE_DIR = Path(PATH_OUTPUT) / 'cache_landkreis_buildingtypes_NEW_BUILDINGS'
PATH_CACHE_STATS = PATH_CACHE_DIR / 'landkreis_buildingtype_stats_NEW_BUILDINGS.csv'
PATH_CACHE_GPKG = PATH_CACHE_DIR / 'landkreis_buildingtype_map_NEW_BUILDINGS.gpkg'
CACHE_LAYER_NAME = 'landkreis_stats'

# Control
REBUILD_CACHE = False
MAKE_MAP = True
MAKE_BAR = True

# Bavaria filter: ARS numbers starting with 09
BAYERN_ARS_PREFIX = '09'

# Color scheme
COLOR_MAP_TYPES = {
    'SFH-DB': "#82cbec",            # Blue
    'SBD': "#febd2b",               # Yellow
    'TB': "#9aab4b",                # Green
    'MFH-AB': "#d94f21",            # Red
    'unclassified res': "#b3b3b3",  # Gray
}

# Template settings
TARGET_CRS = 'EPSG:25832'
CRS_NOTE = 'CRS: EPSG:25832 - ETRS89 / UTM zone 32N'
GRID_STEP_M = 50000
MAP_PADDING_M = 10000
LABEL_OFFSET_M = 9000
LABEL_CLAMP_MARGIN_M = 7000

# ============================================================================
# FUNCTIONS
# ============================================================================

def read_admin_data():
    """Read district data; try desired layer, otherwise fallback."""
    try:
        gdf_admin = gpd.read_file(PATH_LANDKREISE, layer=LAYER_LANDKREISE)
        print(f"  Layer loaded: {LAYER_LANDKREISE}")
        return gdf_admin
    except Exception as e:
        print(f"  Notice: Layer '{LAYER_LANDKREISE}' not readable ({e})")
        print("  Fallback: reading default layer from file")
        return gpd.read_file(PATH_LANDKREISE)


def load_data():
    """Load district layer for Bavaria and filtered building data."""
    print("Loading administrative data (district layer)...")
    gdf_admin = read_admin_data()
    print(f"  Total units: {len(gdf_admin)}")

    # Find ARS column
    if COL_ARS in gdf_admin.columns:
        ars_col = COL_ARS
    else:
        available_cols = [c for c in gdf_admin.columns if 'ARS' in c or 'ars' in c]
        if not available_cols:
            raise ValueError('No ARS column found.')
        ars_col = available_cols[0]
        print(f"  Using ARS column: {ars_col}")

    # Filter Bavaria
    gdf_bayern = gdf_admin[gdf_admin[ars_col].astype(str).str.startswith(BAYERN_ARS_PREFIX)].copy()
    gdf_bayern = gdf_bayern.reset_index(drop=True)
    print(f"  Bavaria districts: {len(gdf_bayern)}")

    # District key: first 5 characters of ARS
    gdf_bayern[COL_LK_KEY] = gdf_bayern[ars_col].astype(str).str[:5]

    # Name column for map labels
    gdf_bayern[COL_LK_NAME] = gdf_bayern[COL_GEN].astype(str) if COL_GEN in gdf_bayern.columns else gdf_bayern[COL_LK_KEY]

    # Columns to carry through dissolve
    keep_cols = [COL_LK_KEY, COL_LK_NAME, COL_GEOMETRY]
    aggfunc = {COL_LK_NAME: 'first'}
    if COL_GEN in gdf_bayern.columns:
        keep_cols.append(COL_GEN)
        aggfunc[COL_GEN] = 'first'
    if COL_EWZ in gdf_bayern.columns:
        keep_cols.append(COL_EWZ)
        aggfunc[COL_EWZ] = 'sum'

    gdf_lk = gdf_bayern[keep_cols].dissolve(by=COL_LK_KEY, aggfunc=aggfunc, as_index=False)
    print(f"  Unique districts in Bavaria: {len(gdf_lk)}")
    print(f"  Columns: {list(gdf_lk.columns)}")

    bbox = tuple(gdf_lk.total_bounds)
    print(f"  Bavaria Bounding Box: {bbox}")

    print("\nLoading building data (filtered to Bavaria bbox)...")
    gdf_buildings = gpd.read_file(
        PATH_BUILDINGS,
        bbox=bbox,
        columns=[COL_RES_SUBCLASS]
    )
    print(f"  Buildings loaded: {len(gdf_buildings)}")

    if gdf_buildings.crs != gdf_lk.crs:
        print(f"  Converting building CRS: {gdf_buildings.crs} -> {gdf_lk.crs}")
        gdf_buildings = gdf_buildings.to_crs(gdf_lk.crs)

    return gdf_buildings, gdf_lk


def spatial_join_buildings_to_landkreise(gdf_buildings, gdf_lk):
    """Spatial join: buildings to districts."""
    print("\nSpatial join at district level...")
    gdf_joined = gpd.sjoin(
        gdf_buildings[[COL_GEOMETRY, COL_RES_SUBCLASS]],
        gdf_lk[[COL_LK_KEY, COL_LK_NAME, COL_GEOMETRY]],
        how='inner',
        predicate='intersects'
    )
    gdf_joined = gdf_joined[~gdf_joined.index.duplicated(keep='first')]
    print(f"  Buildings joined to district: {len(gdf_joined)}")
    return gdf_joined


def compute_stats_per_landkreis(gdf_joined):
    """Compute absolute + percentage values per building type per district + dominant type."""
    print("\nComputing metrics per district...")

    stats = gdf_joined.groupby([COL_LK_KEY, COL_RES_SUBCLASS]).size().reset_index(name='count')
    stats_pivot = stats.pivot_table(
        index=COL_LK_KEY,
        columns=COL_RES_SUBCLASS,
        values='count',
        fill_value=0
    ).reset_index()
    stats_pivot.columns.name = None

    # Ensure all types are present
    for t in BUILDING_TYPES:
        if t not in stats_pivot.columns:
            stats_pivot[t] = 0

    type_cols = BUILDING_TYPES.copy()

    # Absolute total and percentage values
    stats_pivot['total_buildings'] = stats_pivot[type_cols].sum(axis=1)
    for t in type_cols:
        stats_pivot[f'{t}_pct'] = (stats_pivot[t] / stats_pivot['total_buildings'] * 100).round(2)

    # Dominant building type
    stats_pivot['dominant_type'] = stats_pivot[type_cols].idxmax(axis=1)
    stats_pivot['dominant_type_count'] = stats_pivot[type_cols].max(axis=1)
    stats_pivot['dominant_type_pct'] = (stats_pivot['dominant_type_count'] / stats_pivot['total_buildings'] * 100).round(2)

    print(f"  Metrics computed for {len(stats_pivot)} districts")
    return stats_pivot


def create_output_dir():
    output_dir = Path(PATH_OUTPUT)
    output_dir.mkdir(parents=True, exist_ok=True)
    PATH_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    return output_dir


def cache_exists():
    return PATH_CACHE_STATS.exists() and PATH_CACHE_GPKG.exists()


def save_cache(stats_lk, gdf_result):
    stats_lk.to_csv(PATH_CACHE_STATS, index=False, encoding='utf-8-sig')
    gdf_result.to_file(PATH_CACHE_GPKG, layer=CACHE_LAYER_NAME, driver='GPKG')
    print(f"\nCache saved:\n  {PATH_CACHE_STATS}\n  {PATH_CACHE_GPKG}")


def load_cache():
    print("\nLoading data from cache...")
    stats_lk = pd.read_csv(PATH_CACHE_STATS, dtype={COL_LK_KEY: str})
    gdf_result = gpd.read_file(PATH_CACHE_GPKG, layer=CACHE_LAYER_NAME)
    print(f"  Cache loaded: {len(stats_lk)} districts")
    return stats_lk, gdf_result


def build_and_cache_from_raw():
    """Execute heavy pipeline once and cache results."""
    gdf_buildings, gdf_lk = load_data()
    gdf_joined = spatial_join_buildings_to_landkreise(gdf_buildings, gdf_lk)
    del gdf_buildings

    stats_lk = compute_stats_per_landkreis(gdf_joined)
    del gdf_joined

    gdf_result = gdf_lk.merge(stats_lk, on=COL_LK_KEY, how='inner')
    save_cache(stats_lk, gdf_result)
    return stats_lk, gdf_result


def add_matplotlib_grid(ax, bounds, step=GRID_STEP_M):
    """Add UTM coordinate grid with crosses."""
    minx, miny, maxx, maxy = bounds
    x_start = math.ceil(minx / step) * step
    x_end = math.floor(maxx / step) * step
    y_start = math.ceil(miny / step) * step
    y_end = math.floor(maxy / step) * step

    x_major = [x_start + i * step for i in range(int((x_end - x_start) / step) + 1)] if x_start <= x_end else []
    y_major = [y_start + i * step for i in range(int((y_end - y_start) / step) + 1)] if y_start <= y_end else []

    ax.set_xticks(x_major)
    ax.set_yticks(y_major)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{int(round(x / 1000))}'))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f'{int(round(y / 1000))}'))

    if x_major and y_major:
        x_pts, y_pts = [], []
        for xv in x_major:
            for yv in y_major:
                x_pts.append(xv)
                y_pts.append(yv)
        ax.scatter(x_pts, y_pts, marker='+', s=30, linewidths=1.0, color='#8a8a8a', alpha=0.95, zorder=4, clip_on=True)

    ax.tick_params(axis='both', which='major', labelsize=11, length=0, colors='#9B9999')


def add_scientific_frame(ax, gdf_base):
    """Add scientific frame with coordinate labels, grid, and background."""
    ax.set_facecolor('#f1f1f1')
    minx, miny, maxx, maxy = gdf_base.total_bounds
    bounds = (minx - MAP_PADDING_M, miny - MAP_PADDING_M, maxx + MAP_PADDING_M, maxy + MAP_PADDING_M)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.margins(0)
    add_matplotlib_grid(ax, bounds)

    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_autoscale_on(False)

    ax.set_xlabel('Easting (km) - UTM 32N', fontsize=12, color='#555555')
    ax.set_ylabel('Northing (km) - UTM 32N', fontsize=12, color='#555555')
    ax.text(0.01, 0.01, CRS_NOTE, transform=ax.transAxes, ha='left', va='bottom', fontsize=11, color='#555555')

    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
    for side in ['left', 'bottom']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color('#636262')
        ax.spines[side].set_linewidth(0.8)


def add_north_arrow(ax):
    """Add north arrow in upper right corner."""
    if HAS_MMU:
        north_arrow(
            ax=ax,
            location='upper right',
            size='md',
            rotation={'degrees': 0},
            aob={
                'bbox_to_anchor': (0.985, 0.985),
                'bbox_transform': ax.transAxes,
                'pad': 0.06,
                'borderpad': 0.06,
                'facecolor': 'none',
                'edgecolor': 'none',
                'alpha': 1.0,
                'frameon': False,
            },
        )
        return

    ax.annotate(
        'N',
        xy=(0.965, 0.97),
        xytext=(0.965, 0.86),
        xycoords='axes fraction',
        textcoords='axes fraction',
        ha='center',
        va='center',
        fontsize=16,
        fontweight='bold',
        arrowprops=dict(arrowstyle='-|>', color='#222222', linewidth=1.4, shrinkA=0, shrinkB=0)
    )


def add_scale_bar(ax):
    """Add scale bar in lower right corner."""
    if HAS_MMU:
        scale_bar(
            ax=ax,
            location='lower right',
            size='xs',
            style='ticks',
            bar={
                'projection': TARGET_CRS,
                'unit': 'km',
                'max': 50,
                'major_div': 2,
                'minor_div': 1,
                'minor_type': 'none',
                'reverse': False,
            },
            labels={
                'labels': ['0', '25', '50'],
                'style': 'major',
                'loc': 'below',
                'fontsize': 9,
            },
            units={'loc': 'text', 'label': 'km'},
            text={'fontfamily': 'sans-serif', 'fontsize': 9, 'textcolor': '#222222'},
            aob={
                'pad': 0.0,
                'borderpad': 1.0,
                'facecolor': 'none',
                'edgecolor': 'none',
                'alpha': 1.0,
                'frameon': False,
            },
        )
        return

    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    span_x = x1 - x0
    span_y = y1 - y0

    total_km = 50
    segment_km = 25
    bar_len = total_km * 1000
    segment_len = segment_km * 1000

    x_start = x1 - span_x * 0.25
    y_start = y0 + span_y * 0.032

    ax.plot([x_start, x_start + bar_len], [y_start, y_start], color='#222222', linewidth=1.3, zorder=5)
    tick_h = span_y * 0.006
    for x in [x_start, x_start + segment_len, x_start + bar_len]:
        ax.plot([x, x], [y_start - tick_h, y_start + tick_h], color='#222222', linewidth=1.0, zorder=5)

    txt_y = y_start + span_y * 0.011
    ax.text(x_start, txt_y, '0', ha='center', va='bottom', fontsize=9, color='#222222')
    ax.text(x_start + segment_len, txt_y, '25', ha='center', va='bottom', fontsize=9, color='#222222')
    ax.text(x_start + bar_len, txt_y, '50 km', ha='center', va='bottom', fontsize=9, color='#222222')


def add_labels_non_sfh(ax, gdf, offset_m=LABEL_OFFSET_M):
    """Label all districts whose dominant type is not SFH-DB with offset positioning."""
    required_cols = {COL_LK_NAME, 'dominant_type', COL_GEOMETRY}
    if not required_cols.issubset(set(gdf.columns)):
        print("  Notice: Required columns for labels missing")
        return

    gdf_labels = gdf[gdf['dominant_type'] != 'SFH-DB'].copy()
    if gdf_labels.empty:
        print("  Notice: No non-SFH-DB districts available for labeling")
        return

    bounds = gdf.total_bounds
    center_x = (bounds[0] + bounds[2]) / 2
    center_y = (bounds[1] + bounds[3]) / 2

    # Keep label anchors inside plotted map frame (with small inner margin)
    x_min = bounds[0] - MAP_PADDING_M + LABEL_CLAMP_MARGIN_M
    x_max = bounds[2] + MAP_PADDING_M - LABEL_CLAMP_MARGIN_M
    y_min = bounds[1] - MAP_PADDING_M + LABEL_CLAMP_MARGIN_M
    y_max = bounds[3] + MAP_PADDING_M - LABEL_CLAMP_MARGIN_M

    # Build normalized name groups to disambiguate equal names (e.g. city vs district)
    def _norm_name(name):
        n = str(name).strip()
        n = n.replace('Landkreis ', '').replace('Lkr. ', '').replace('Stadtkreis ', '')
        n = n.replace('Kreisfreie Stadt ', '').replace('Landeshauptstadt ', '').replace('Stadt ', '')
        n = n.split(',')[0].strip().lower()
        return n

    type_by_idx = {}
    grouped = {}
    for idx, row in gdf_labels.iterrows():
        key = _norm_name(row[COL_LK_NAME])
        grouped.setdefault(key, []).append((idx, row.geometry.area))

    for key, items in grouped.items():
        if len(items) <= 1:
            continue
        # Largest area is typically district, smallest is typically city
        items_sorted = sorted(items, key=lambda x: x[1])
        for i, (idx, _) in enumerate(items_sorted):
            type_by_idx[idx] = 'stadt' if i == 0 else 'landkreis'

    def _format_label(row):
        raw = str(row[COL_LK_NAME]).strip()
        raw_low = raw.lower()
        base = raw.split(',')[0].strip()

        if row.name in type_by_idx:
            return f"Stadt {base}" if type_by_idx[row.name] == 'stadt' else f"Lkr. {base}"

        if 'landkreis' in raw_low or raw_low.startswith('lkr.'):
            return f"Lkr. {base}"
        if 'stadt' in raw_low or 'landeshauptstadt' in raw_low:
            return f"Stadt {base}"
        return base

    # Collision thresholds in data units
    span_x = (bounds[2] + MAP_PADDING_M) - (bounds[0] - MAP_PADDING_M)
    span_y = (bounds[3] + MAP_PADDING_M) - (bounds[1] - MAP_PADDING_M)
    min_dx = span_x * 0.11
    min_dy = span_y * 0.03
    placed = []

    points = gdf_labels.representative_point()
    for (_, row), p in zip(gdf_labels.iterrows(), points):
        dx = p.x - center_x
        dy = p.y - center_y
        norm = math.hypot(dx, dy)
        if norm == 0:
            dx, dy, norm = 1.0, 1.0, math.sqrt(2)
        ux, uy = dx / norm, dy / norm

        # Try multiple directions; if one direction crosses map bounds, switch direction
        direction_candidates = [
            (ux, uy),
            (-ux, -uy),
            (1.0 if ux >= 0 else -1.0, 0.0),
            (-1.0 if ux >= 0 else 1.0, 0.0),
            (0.0, 1.0 if uy >= 0 else -1.0),
            (0.0, -1.0 if uy >= 0 else 1.0),
        ]

        chosen = None
        for cx, cy in direction_candidates:
            cand_tx = p.x + cx * offset_m
            cand_ty = p.y + cy * offset_m
            if x_min <= cand_tx <= x_max and y_min <= cand_ty <= y_max:
                chosen = (cand_tx, cand_ty, cx)
                break

        if chosen is None:
            cand_tx = p.x - ux * offset_m
            cand_ty = p.y - uy * offset_m
            cand_tx = min(max(cand_tx, x_min), x_max)
            cand_ty = min(max(cand_ty, y_min), y_max)
            chosen = (cand_tx, cand_ty, -ux)

        tx, ty, dir_x = chosen

        # Iteratively move labels until there is no overlap with previously placed labels
        label_side = 1 if dir_x >= 0 else -1
        for _ in range(40):
            overlap = any((abs(tx - px) < min_dx and abs(ty - py) < min_dy) for px, py in placed)
            if not overlap:
                break
            ty += (min_dy * 0.55) if ty <= center_y else (-min_dy * 0.55)
            ty = min(max(ty, y_min), y_max)
            tx += label_side * (min_dx * 0.20)
            tx = min(max(tx, x_min), x_max)

        # If still overlapping after retries, skip this label to avoid unreadable collisions
        if any((abs(tx - px) < min_dx and abs(ty - py) < min_dy) for px, py in placed):
            continue

        ha = 'left' if tx >= p.x else 'right'
        label_text = _format_label(row)

        # Avoid left/right clipping by adapting text anchor and x-position
        approx_char_w = span_x * 0.0065
        text_w = max(1, len(label_text)) * approx_char_w
        pad_x = span_x * 0.006
        if tx - text_w < x_min + pad_x:
            ha = 'left'
            tx = min(max(tx, x_min + pad_x), x_max - pad_x)
        elif tx + text_w > x_max - pad_x:
            ha = 'right'
            tx = max(min(tx, x_max - pad_x), x_min + pad_x)

        placed.append((tx, ty))

        ax.annotate(
            label_text,
            xy=(p.x, p.y),
            xytext=(tx, ty),
            textcoords='data',
            ha=ha,
            va='center',
            fontsize=10,
            color='#1f1f1f',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.7),
            arrowprops=dict(arrowstyle='-', color='#8a8a8a', lw=0.7),
            zorder=6,
            clip_on=True,
        )


def add_legend_bottom_right(ax, handles, title):
    """Add legend outside on the right side."""
    ax.legend(
        handles=handles,
        title=title,
        loc='lower left',
        bbox_to_anchor=(1.01, 0.02),
        borderaxespad=0.0,
        frameon=False,
        fontsize=13,
        title_fontsize=14,
        handlelength=2.0,
        labelspacing=0.6,
        borderpad=0.0,
    )


def plot_dominant_type_map(gdf, output_dir):
    """Map: Dominant building type per district with template theme."""
    print("\nCreating map: Dominant building type (district level)...")

    fig, ax = plt.subplots(figsize=(8.6, 9.4))
    fig.subplots_adjust(right=0.80, left=0.08, top=0.92, bottom=0.10)

    colors = gdf['dominant_type'].map(COLOR_MAP_TYPES).fillna('#bdbdbd')
    gdf.plot(ax=ax, color=colors, edgecolor='white', linewidth=0.6, zorder=2)

    present_types = [t for t in BUILDING_TYPES if t in gdf['dominant_type'].unique()]
    legend_elements = [
        mpatches.Patch(facecolor=COLOR_MAP_TYPES[t], edgecolor='white', label=t)
        for t in present_types
    ]

    add_labels_non_sfh(ax, gdf)
    ax.set_title('Dominant Residential Building Types of Buildings Constructed Since 2015 at the District Level in Bavaria', fontweight='bold', fontsize=18, pad=14)
    add_scientific_frame(ax, gdf)
    add_north_arrow(ax)
    add_scale_bar(ax)
    add_legend_bottom_right(ax, legend_elements, 'Dominant\nBuilding Type')

    out_file = output_dir / 'dominant_type_bayern_landkreise_NEW_BUILDINGS.jpg'
    plt.savefig(out_file, bbox_inches='tight', facecolor='white', format='jpg')
    print(f"  Saved: {out_file}")

    plt.close()

# ============================================================================
# MAIN PROGRAM
# ============================================================================

print("=" * 72)
print("RESIDENTIAL BUILDING TYPES BAVARIA - DISTRICT LEVEL")
print("=" * 72)

output_dir = create_output_dir()

# 1) Load data from cache or recompute
if (not REBUILD_CACHE) and cache_exists():
    stats_lk, gdf_result = load_cache()
else:
    print("\nCache missing or REBUILD_CACHE=True -> recomputing...")
    stats_lk, gdf_result = build_and_cache_from_raw()

# 2) Generate plots (fast, without heavy computation)
if MAKE_MAP:
    plot_dominant_type_map(gdf_result, output_dir)

# 3) CSV export with absolute + percentage values
csv_file = output_dir / 'landkreis_buildingtype_stats_NEW_BUILDINGS.csv'

stats_export_cols = ['total_buildings', 'dominant_type', 'dominant_type_count', 'dominant_type_pct']
for t in BUILDING_TYPES:
    stats_export_cols.append(t)
for t in BUILDING_TYPES:
    stats_export_cols.append(f'{t}_pct')

export_df = stats_lk[[COL_LK_KEY] + stats_export_cols].copy()

# Merge name and population from gdf_result
for col in [COL_GEN, COL_EWZ]:
    if col in gdf_result.columns:
        export_df = export_df.merge(
            gdf_result[[COL_LK_KEY, col]].drop_duplicates(subset=COL_LK_KEY),
            on=COL_LK_KEY,
            how='left'
        )

# Rename LK_KEY -> ARS, preserve leading zeros
export_df = export_df.rename(columns={COL_LK_KEY: 'ARS'})
export_df['ARS'] = export_df['ARS'].astype(str).str.zfill(5)

# Column order: ARS, name, population, then stats
front_cols = ['ARS']
for col in [COL_GEN, COL_EWZ]:
    if col in export_df.columns:
        front_cols.append(col)
export_df = export_df[front_cols + stats_export_cols]

export_df.to_csv(csv_file, index=False, encoding='utf-8-sig')
print(f"\nCSV saved: {csv_file}")

print("\nDone. Output saved to:")
print(output_dir)
print("\nNote: For plot adjustments, set REBUILD_CACHE=False; then only cache is loaded.")

RESIDENTIAL BUILDING TYPES BAVARIA - DISTRICT LEVEL

Loading data from cache...
  Cache loaded: 96 districts

Creating map: Dominant building type (district level)...
  Saved: C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen\dominant_type_bayern_landkreise_NEW_BUILDINGS.jpg

CSV saved: C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen\landkreis_buildingtype_stats_NEW_BUILDINGS.csv

Done. Output saved to:
C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen

Note: For plot adjustments, set REBUILD_CACHE=False; then only cache is loaded.
